# Liveliness Signal Backtest (2019+ Only)

Excluding the anomalous 2015-2017 cycle where the coefficient was opposite.

## Contents
1. Setup and Data Loading (2019+)
2. Signal Generation
3. Basic Backtest
4. Parameter Optimization
5. Walk-Forward Validation
6. Final Results

---
## 1. Setup and Data Loading (2019+ Only)

In [ ]:
import pandas as pd
import numpy as np
import vectorbt as vbt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Disable widgets
vbt.settings.plotting['use_widgets'] = False
vbt.settings.array_wrapper['freq'] = 'D'
vbt.settings.portfolio['init_cash'] = 100_000

print(f"VectorBT version: {vbt.__version__}")

In [ ]:
# Load data
DATA_DIR = Path("../data/raw")

liveliness = pd.read_parquet(DATA_DIR / "liveliness.parquet")
price = pd.read_parquet(DATA_DIR / "price.parquet")

liveliness = liveliness.rename(columns={"value": "liveliness"}).set_index("time")
price = price.rename(columns={"value": "price"}).set_index("time")

df = liveliness.join(price, how='inner').sort_index()

# ========================================
# FILTER TO 2019+ ONLY
# ========================================
START_DATE = '2018-12-15'  # Start of 2019 bull cycle
df = df[df.index >= START_DATE]

close = df['price']
liveliness_series = df['liveliness']

print(f"Data loaded: {len(df)} rows")
print(f"Date range: {df.index.min().date()} to {df.index.max().date()}")
print(f"\nPrice range: ${close.min():,.0f} - ${close.max():,.0f}")
print(f"Liveliness range: {liveliness_series.min():.4f} - {liveliness_series.max():.4f}")
print(f"\n⚠️  Excluding 2015-2017 cycle (opposite coefficient)")

In [ ]:
# Quick visualization
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=['Bitcoin Price (2019+)', 'Liveliness'])

fig.add_trace(go.Scatter(x=close.index, y=close, name='Price'), row=1, col=1)
fig.add_trace(go.Scatter(x=liveliness_series.index, y=liveliness_series, name='Liveliness'), row=2, col=1)

fig.update_layout(height=600, showlegend=False)
fig.show()

---
## 2. Signal Generation

From regression: **Negative coefficient** in all post-2018 cycles → Buy when liveliness is LOW

In [ ]:
# Signal parameters
THRESHOLD = 0.49  # Initial threshold from regression

# Generate signals
in_signal = liveliness_series < THRESHOLD
entries = in_signal & ~in_signal.shift(1).fillna(False)
exits = ~in_signal & in_signal.shift(1).fillna(False)

print(f"Signal: Buy when liveliness < {THRESHOLD}")
print(f"\nEntry signals: {entries.sum()}")
print(f"Exit signals: {exits.sum()}")
print(f"Days in signal: {in_signal.sum()} ({in_signal.mean()*100:.1f}%)")

In [ ]:
# Visualize signals
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=['Price with Entry/Exit', 'Liveliness with Threshold'])

fig.add_trace(go.Scatter(x=close.index, y=close, name='Price', line=dict(color='blue')), row=1, col=1)

# Entries
entry_prices = close[entries]
fig.add_trace(go.Scatter(x=entry_prices.index, y=entry_prices, mode='markers',
                         marker=dict(symbol='triangle-up', size=12, color='green'),
                         name='Entry'), row=1, col=1)

# Exits
exit_prices = close[exits]
fig.add_trace(go.Scatter(x=exit_prices.index, y=exit_prices, mode='markers',
                         marker=dict(symbol='triangle-down', size=12, color='red'),
                         name='Exit'), row=1, col=1)

# Liveliness
fig.add_trace(go.Scatter(x=liveliness_series.index, y=liveliness_series, 
                         name='Liveliness', line=dict(color='orange')), row=2, col=1)
fig.add_hline(y=THRESHOLD, line_dash='dash', line_color='red', row=2, col=1)

fig.update_layout(height=700)
fig.show()

---
## 3. Basic Backtest

In [ ]:
# Run backtest
pf = vbt.Portfolio.from_signals(
    close=close,
    entries=entries,
    exits=exits,
    init_cash=100_000,
    fees=0.001,
    slippage=0.001,
    freq='D'
)

# Buy & Hold benchmark
pf_hold = vbt.Portfolio.from_holding(close, init_cash=100_000, freq='D')

print("STRATEGY STATS (2019+):")
print("="*50)
print(pf.stats())

In [ ]:
# Compare to Buy & Hold
print("\n" + "="*60)
print("STRATEGY vs BUY & HOLD (2019+)")
print("="*60)
print(f"{'Metric':<25} {'Strategy':>15} {'Buy & Hold':>15}")
print("-"*60)
print(f"{'Total Return':<25} {pf.total_return()*100:>14.1f}% {pf_hold.total_return()*100:>14.1f}%")
print(f"{'Sharpe Ratio':<25} {pf.sharpe_ratio():>15.2f} {pf_hold.sharpe_ratio():>15.2f}")
print(f"{'Sortino Ratio':<25} {pf.sortino_ratio():>15.2f} {pf_hold.sortino_ratio():>15.2f}")
print(f"{'Max Drawdown':<25} {pf.max_drawdown()*100:>14.1f}% {pf_hold.max_drawdown()*100:>14.1f}%")
print(f"{'Win Rate':<25} {pf.trades.win_rate()*100:>14.1f}% {'N/A':>15}")
print(f"{'Total Trades':<25} {pf.trades.count():>15} {'1':>15}")

In [ ]:
# Plot equity curves
fig = go.Figure()

strat_equity = pf.value()
hold_equity = pf_hold.value()

fig.add_trace(go.Scatter(x=strat_equity.index, y=strat_equity, name='Liveliness Strategy'))
fig.add_trace(go.Scatter(x=hold_equity.index, y=hold_equity, name='Buy & Hold'))

fig.update_layout(
    title='Equity Curve: Strategy vs Buy & Hold (2019+)',
    yaxis_title='Portfolio Value ($)',
    height=500
)
fig.show()

---
## 4. Parameter Optimization (Grid Search)

In [ ]:
# Test range of thresholds
thresholds = np.linspace(
    liveliness_series.quantile(0.05),
    liveliness_series.quantile(0.95),
    25
)

print(f"Testing {len(thresholds)} thresholds")
print(f"Range: {thresholds.min():.4f} to {thresholds.max():.4f}")

In [ ]:
# Generate signals for all thresholds
entries_matrix = pd.DataFrame(index=close.index)
exits_matrix = pd.DataFrame(index=close.index)

for thresh in thresholds:
    in_sig = liveliness_series < thresh
    entries_matrix[thresh] = in_sig & ~in_sig.shift(1).fillna(False)
    exits_matrix[thresh] = ~in_sig & in_sig.shift(1).fillna(False)

# Run all backtests at once
pf_opt = vbt.Portfolio.from_signals(
    close=close,
    entries=entries_matrix,
    exits=exits_matrix,
    init_cash=100_000,
    fees=0.001,
    slippage=0.001,
    freq='D'
)

# Collect results
results = pd.DataFrame({
    'threshold': thresholds,
    'total_return': pf_opt.total_return().values,
    'sharpe': pf_opt.sharpe_ratio().values,
    'sortino': pf_opt.sortino_ratio().values,
    'max_dd': pf_opt.max_drawdown().values,
    'win_rate': pf_opt.trades.win_rate().values,
    'n_trades': pf_opt.trades.count().values
})

# Add buy & hold comparison
hold_return = pf_hold.total_return()
results['excess_return'] = results['total_return'] - hold_return

results

In [ ]:
# Plot optimization results
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=['Sharpe Ratio', 'Excess Return vs B&H',
                                   'Max Drawdown', 'Number of Trades'])

# Sharpe
fig.add_trace(go.Scatter(x=results['threshold'], y=results['sharpe'],
                         mode='lines+markers', name='Sharpe'), row=1, col=1)
fig.add_hline(y=pf_hold.sharpe_ratio(), line_dash='dash', line_color='red', row=1, col=1)

# Excess Return
colors = ['green' if x > 0 else 'red' for x in results['excess_return']]
fig.add_trace(go.Bar(x=results['threshold'], y=results['excess_return']*100,
                     marker_color=colors, name='Excess'), row=1, col=2)
fig.add_hline(y=0, line_dash='dash', row=1, col=2)

# Max DD
fig.add_trace(go.Scatter(x=results['threshold'], y=results['max_dd']*100,
                         mode='lines+markers', name='Max DD'), row=2, col=1)
fig.add_hline(y=pf_hold.max_drawdown()*100, line_dash='dash', line_color='red', row=2, col=1)

# Trades
fig.add_trace(go.Bar(x=results['threshold'], y=results['n_trades'], name='Trades'), row=2, col=2)

fig.update_layout(height=700, showlegend=False,
                  title_text='Grid Search Results (Red dashed = Buy & Hold)')
fig.show()

# Smoothness
sharpe_vals = results['sharpe'].fillna(0).values
sharpe_diffs = np.diff(sharpe_vals)
smoothness = np.std(sharpe_diffs) / (np.mean(np.abs(sharpe_vals)) + 1e-6)

print(f"\nSmoothness score: {smoothness:.2f} ({'✓ Smooth' if smoothness < 0.5 else '✗ Spiky'})")

In [ ]:
# Best threshold
best_idx = results['sharpe'].idxmax()
best = results.loc[best_idx]

print("\n" + "="*50)
print("OPTIMAL THRESHOLD (2019+)")
print("="*50)
print(f"Threshold: {best['threshold']:.4f}")
print(f"Sharpe: {best['sharpe']:.2f}")
print(f"Return: {best['total_return']*100:.1f}%")
print(f"Excess vs B&H: {best['excess_return']*100:+.1f}%")
print(f"Max DD: {best['max_dd']*100:.1f}%")
print(f"Trades: {best['n_trades']:.0f}")

---
## 5. Walk-Forward Validation

In [ ]:
# Walk-forward parameters
TRAIN_DAYS = 365  # 1 year training
TEST_DAYS = 90    # 3 months testing
STEP_DAYS = 90    # Roll every 3 months

print(f"Walk-Forward Configuration:")
print(f"  Training: {TRAIN_DAYS} days")
print(f"  Testing: {TEST_DAYS} days")
print(f"  Step: {STEP_DAYS} days")

In [ ]:
# Walk-forward loop
wf_results = []

total_days = len(close)
n_folds = (total_days - TRAIN_DAYS) // STEP_DAYS

for fold in range(n_folds):
    train_start = fold * STEP_DAYS
    train_end = train_start + TRAIN_DAYS
    test_start = train_end
    test_end = min(test_start + TEST_DAYS, total_days)
    
    if test_end <= test_start:
        break
    
    # Get data slices
    train_close = close.iloc[train_start:train_end]
    train_live = liveliness_series.iloc[train_start:train_end]
    test_close = close.iloc[test_start:test_end]
    test_live = liveliness_series.iloc[test_start:test_end]
    
    # TRAIN: Find best threshold
    best_sharpe = -np.inf
    best_thresh = None
    
    for thresh in thresholds:
        in_sig = train_live < thresh
        ent = in_sig & ~in_sig.shift(1).fillna(False)
        ext = ~in_sig & in_sig.shift(1).fillna(False)
        
        if ent.sum() < 2:
            continue
        
        try:
            pf_train = vbt.Portfolio.from_signals(
                close=train_close, entries=ent, exits=ext,
                init_cash=100_000, fees=0.001, freq='D'
            )
            sharpe = pf_train.sharpe_ratio()
            if not np.isnan(sharpe) and sharpe > best_sharpe:
                best_sharpe = sharpe
                best_thresh = thresh
        except:
            continue
    
    if best_thresh is None:
        continue
    
    # TEST: Apply best threshold
    in_sig_test = test_live < best_thresh
    ent_test = in_sig_test & ~in_sig_test.shift(1).fillna(False)
    ext_test = ~in_sig_test & in_sig_test.shift(1).fillna(False)
    
    try:
        pf_test = vbt.Portfolio.from_signals(
            close=test_close, entries=ent_test, exits=ext_test,
            init_cash=100_000, fees=0.001, freq='D'
        )
        pf_hold_test = vbt.Portfolio.from_holding(test_close, init_cash=100_000, freq='D')
        
        test_return = pf_test.total_return()
        hold_return = pf_hold_test.total_return()
        test_sharpe = pf_test.sharpe_ratio()
    except:
        continue
    
    wf_results.append({
        'fold': fold,
        'test_start': close.index[test_start].strftime('%Y-%m-%d'),
        'test_end': close.index[test_end-1].strftime('%Y-%m-%d'),
        'threshold': best_thresh,
        'train_sharpe': best_sharpe,
        'test_sharpe': test_sharpe if not np.isnan(test_sharpe) else 0,
        'test_return': test_return,
        'hold_return': hold_return,
        'excess_return': test_return - hold_return,
        'beat_hold': test_return > hold_return
    })
    
    status = '✓' if test_return > hold_return else '✗'
    print(f"Fold {fold}: {close.index[test_start].strftime('%Y-%m')} | "
          f"Thresh: {best_thresh:.3f} | "
          f"Strat: {test_return*100:+.1f}% | "
          f"B&H: {hold_return*100:+.1f}% | {status}")

wf_df = pd.DataFrame(wf_results)
print(f"\nCompleted {len(wf_df)} folds")

In [ ]:
# Walk-forward summary
print("\n" + "="*60)
print("WALK-FORWARD RESULTS (2019+)")
print("="*60)

if len(wf_df) > 0:
    print(f"\n{'Metric':<30} {'Value':>15}")
    print("-"*50)
    print(f"{'Folds Tested':<30} {len(wf_df):>15}")
    print(f"{'Avg Test Sharpe':<30} {wf_df['test_sharpe'].mean():>15.2f}")
    print(f"{'Avg Strategy Return':<30} {wf_df['test_return'].mean()*100:>14.1f}%")
    print(f"{'Avg Buy&Hold Return':<30} {wf_df['hold_return'].mean()*100:>14.1f}%")
    print(f"{'Avg Excess Return':<30} {wf_df['excess_return'].mean()*100:>+14.1f}%")
    print(f"{'Beat Buy&Hold Rate':<30} {wf_df["beat_hold"].mean()*100:>14.1f}%")
    print(f"{'Threshold Stability (std)':<30} {wf_df['threshold'].std():>15.4f}")
else:
    print("No valid folds completed")

In [ ]:
# Visualize walk-forward
if len(wf_df) > 0:
    fig = make_subplots(rows=2, cols=2,
                        subplot_titles=['Strategy vs B&H Returns', 'Excess Return',
                                       'Selected Threshold', 'Cumulative Excess'])

    # Returns comparison
    fig.add_trace(go.Bar(x=wf_df['fold'], y=wf_df['test_return']*100, name='Strategy'), row=1, col=1)
    fig.add_trace(go.Scatter(x=wf_df['fold'], y=wf_df['hold_return']*100, 
                             mode='markers', marker=dict(size=12, symbol='diamond', color='red'),
                             name='B&H'), row=1, col=1)

    # Excess return
    colors = ['green' if x else 'red' for x in wf_df['beat_hold']]
    fig.add_trace(go.Bar(x=wf_df['fold'], y=wf_df['excess_return']*100, 
                         marker_color=colors, name='Excess'), row=1, col=2)
    fig.add_hline(y=0, line_dash='dash', row=1, col=2)

    # Threshold
    fig.add_trace(go.Scatter(x=wf_df['fold'], y=wf_df['threshold'],
                             mode='lines+markers', name='Threshold'), row=2, col=1)

    # Cumulative excess
    cum_excess = (1 + wf_df['excess_return']).cumprod() - 1
    fig.add_trace(go.Scatter(x=wf_df['fold'], y=cum_excess*100,
                             mode='lines+markers', name='Cum Excess'), row=2, col=2)
    fig.add_hline(y=0, line_dash='dash', row=2, col=2)

    fig.update_layout(height=700, showlegend=True)
    fig.show()

In [ ]:
# Full results table
if len(wf_df) > 0:
    display_df = wf_df.copy()
    display_df['test_return'] = (display_df['test_return'] * 100).round(1).astype(str) + '%'
    display_df['hold_return'] = (display_df['hold_return'] * 100).round(1).astype(str) + '%'
    display_df['excess_return'] = (display_df['excess_return'] * 100).round(1).astype(str) + '%'
    display_df['threshold'] = display_df['threshold'].round(4)
    display_df['beat_hold'] = display_df['beat_hold'].map({True: '✓', False: '✗'})
    print(display_df.to_string(index=False))

---
## 6. Final Summary

In [ ]:
# Final summary
print("\n" + "="*70)
print("LIVELINESS SIGNAL - FINAL RESULTS (2019+ ONLY)")
print("="*70)

print(f"\n📊 SIGNAL")
print(f"   Buy when liveliness < {best['threshold']:.4f}")
print(f"   (Low liveliness = HODLers accumulating)")

print(f"\n📈 IN-SAMPLE (2019+)")
print(f"   Sharpe: {best['sharpe']:.2f}")
print(f"   Return: {best['total_return']*100:.1f}%")
print(f"   vs B&H: {best['excess_return']*100:+.1f}%")

if len(wf_df) > 0:
    print(f"\n🔍 OUT-OF-SAMPLE (Walk-Forward)")
    print(f"   Avg Sharpe: {wf_df['test_sharpe'].mean():.2f}")
    print(f"   Avg Return: {wf_df['test_return'].mean()*100:.1f}%")
    print(f"   Beat B&H: {wf_df['beat_hold'].mean()*100:.0f}% of periods")
    
    verdict = "✓ SIGNAL WORKS" if wf_df['beat_hold'].mean() > 0.5 else "✗ SIGNAL DOESN'T ADD VALUE"
    print(f"\n🎯 VERDICT: {verdict}")

print("\n" + "="*70)

In [ ]:
# Save results
results_summary = {
    'signal': 'liveliness',
    'direction': 'below',
    'threshold': float(best['threshold']),
    'data_start': '2019+',
    'in_sample_sharpe': float(best['sharpe']) if not np.isnan(best['sharpe']) else None,
    'in_sample_return': float(best['total_return']),
    'in_sample_excess': float(best['excess_return']),
    'oos_sharpe': float(wf_df['test_sharpe'].mean()) if len(wf_df) > 0 else None,
    'oos_return': float(wf_df['test_return'].mean()) if len(wf_df) > 0 else None,
    'beat_hold_pct': float(wf_df['beat_hold'].mean()) if len(wf_df) > 0 else None,
    'smoothness': float(smoothness),
    'n_folds': len(wf_df)
}

import json
with open('../data/liveliness_backtest_2019plus.json', 'w') as f:
    json.dump(results_summary, f, indent=2)
    
print("Results saved to ../data/liveliness_backtest_2019plus.json")
print(json.dumps(results_summary, indent=2))